### **QLoRA, DPO y ORPO**


Este cuaderno resume tres ideas clave del bloque de adaptación y alineamiento de LLMs:

1. **QLoRA** para ajustar un modelo grande con bajo consumo de memoria.
2. **DPO** para entrenar con preferencias sin pasar por un reward model explícito.
3. **ORPO** para unificar ajuste supervisado y preferencia en una sola función objetivo.

La meta es que puedas:
- entender la intuición matemática,
- ver implementaciones pequeñas en PyTorch,
- tener plantillas realistas para Hugging Face y TRL,
- comparar cuándo conviene usar cada método.

> Sugerencia de uso: primero lee los bloques conceptuales, luego ejecuta los ejemplos, y al final adapta las plantillas a tu modelo.

#Notas de REWard MODEL
En aprendizaje por refuerzo (Reinforcement Learning), reward significa recompensa.

Es simplemente un número que indica qué tan buena fue una acción.

Por ejemplo, si un robot debe encontrar la salida de un laberinto:

Encuentra la salida → recompensa = +100
Choca con una pared → recompensa = -10
Da un paso cualquiera → recompensa = -1

El robot aprende a maximizar esa recompensa.
Aquí aparece el algoritmo PPO (Proximal Policy Optimization), que usa esas recompensas para modificar el modelo.
#Resumen
Reward Model: un modelo adicional que califica la calidad de las respuestas según las preferencias humanas.
RLHF clásico: Modelo base → Reward Model → PPO → Modelo alineado.
DPO: elimina el Reward Model y PPO, y aprende directamente a partir de pares de respuestas donde una es preferida sobre la otra.

En ORPO la pérdida combina ambos objetivos en una sola expresión matemática. En cada actualización del modelo, el optimizador intenta:

hacer más probable la respuesta preferida;
hacer menos probable la respuesta rechazada.

No es necesario entrenar primero con una pérdida y luego con otra.

#### **Mapa del cuaderno**

- QLoRA:
  - compresión y memoria,
  - cuantización de 4 bits,
  - adaptación LoRA sobre pesos congelados,
  - plantilla práctica.

- DPO:
  - datos de preferencias,
  - pérdida objetivo,
  - ejemplo elemental,
  - plantilla práctica.

- ORPO:
  - idea central,
  - pérdida unificada,
  - ejemplo elemental,
  - plantilla práctica.

- Cierre:
  - tabla comparativa,
  - recomendaciones de uso,
  - preguntas de sustentación oral.

#### **Importaciones**

Este bloque reúne las librerías ligeras que usaremos en los ejemplos.  
Las plantillas de entrenamiento real aparecerán después en celdas separadas.

In [2]:
import math
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
random.seed(7)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


### **SFT y PEFT antes de QLoRA**

Antes de entrar a QLoRA conviene fijar dos ideas:

- **SFT** ajusta el modelo con ejemplos supervisados del tipo instrucción -> respuesta.
- **PEFT** busca adaptar el modelo sin actualizar todos sus parámetros.

QLoRA se entiende mejor si primero ubicamos estas dos capas conceptuales.


#### **SFT como punto de partida**

En **supervised fine-tuning** tomamos un modelo preentrenado y lo ajustamos con pares del tipo:

- `prompt`
- `respuesta_objetivo`

La pérdida típica es la de modelado causal sobre la respuesta esperada.  
Si el modelo produce probabilidades condicionales $p_\theta(y_t \mid x, y_{<t})$, la pérdida puede escribirse como:

$$
\mathcal{L}_{\mathrm{SFT}}(\theta)
=
-\sum_{t=1}^{T}\log p_\theta(y_t \mid x, y_{<t})
$$

SFT mejora obediencia a instrucciones, formato y estilo de respuesta, pero **no garantiza** que el modelo refleje preferencias humanas finas.


#### **PEFT como familia de métodos**

Si el modelo es grande, actualizar todos los pesos puede ser costoso.  
Ahí aparece **PEFT** (**Parameter-Efficient Fine-Tuning**), que busca modificar solo una parte pequeña del sistema.

Ejemplos típicos:

- **Adapters**
- **LoRA**
- **Prefix tuning**
- **Prompt tuning**

La pregunta central es:

> ¿Cómo adaptar el comportamiento del modelo sin pagar el costo completo de un full fine-tuning?

QLoRA es una respuesta muy efectiva a esa pregunta.


#### **Mini ejemplo de formato para SFT**

El siguiente bloque arma un pequeño dataset de instrucciones.  
No entrenaremos un LLM completo aquí.  
Solo mostraremos la estructura de datos que luego usaríamos en un `Trainer` o `SFTTrainer`.


In [3]:
sft_examples = [
    {
        "instruction": "Explica qué es LoRA en dos frases.",
        "input": "",
        "output": "LoRA aprende una actualización de bajo rango sobre ciertos pesos del modelo. Esto permite ajustar el modelo con menos parámetros entrenables y menor costo de memoria."
    },
    {
        "instruction": "Define QLoRA de forma breve.",
        "input": "",
        "output": "QLoRA combina cuantización de 4 bits para el modelo base con adaptadores LoRA entrenables. Así reduce memoria sin renunciar a una adaptación efectiva."
    },
    {
        "instruction": "Compara DPO y RLHF en una idea.",
        "input": "",
        "output": "DPO optimiza preferencias de forma directa usando pares elegida rechazada, mientras que RLHF clásico suele pasar por reward model y PPO."
    },
]

def format_sft_example(example):
    instruction = example["instruction"].strip()
    input_text = example["input"].strip()
    output = example["output"].strip()
    if input_text:
        prompt = f"### Instrucción:\n{instruction}\n\n### Entrada:\n{input_text}\n\n### Respuesta:\n{output}"
    else:
        prompt = f"### Instrucción:\n{instruction}\n\n### Respuesta:\n{output}"
    return prompt

for ex in sft_examples:
    print(format_sft_example(ex))
    print("-" * 80)


### Instrucción:
Explica qué es LoRA en dos frases.

### Respuesta:
LoRA aprende una actualización de bajo rango sobre ciertos pesos del modelo. Esto permite ajustar el modelo con menos parámetros entrenables y menor costo de memoria.
--------------------------------------------------------------------------------
### Instrucción:
Define QLoRA de forma breve.

### Respuesta:
QLoRA combina cuantización de 4 bits para el modelo base con adaptadores LoRA entrenables. Así reduce memoria sin renunciar a una adaptación efectiva.
--------------------------------------------------------------------------------
### Instrucción:
Compara DPO y RLHF en una idea.

### Respuesta:
DPO optimiza preferencias de forma directa usando pares elegida rechazada, mientras que RLHF clásico suele pasar por reward model y PPO.
--------------------------------------------------------------------------------


#### **Plantilla corta para SFT con Hugging Face**

La lógica práctica suele ser:

1. cargar tokenizer y modelo,
2. preparar un dataset con texto formateado,
3. tokenizar,
4. entrenar con `Trainer` o `SFTTrainer`.

La siguiente celda es solo una plantilla corta.


In [5]:
# Plantilla orientativa para SFT
# !pip install transformers datasets trl accelerate peft

from transformers import AutoTokenizer, AutoModelForCausalLM

# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# modelo = AutoModelForCausalLM.from_pretrained(model_name)

# formatted_texts = [format_sft_example(ex) for ex in sft_examples]
# tokenized = tokenizer(formatted_texts, truncation=True, padding=True, return_tensors="pt")

# Con TRL también podrías usar:
# from trl import SFTTrainer, SFTConfig


### **QLoRA**

QLoRA combina dos ideas:

1. **Cuantizar** los pesos base del modelo a baja precisión.
2. **Entrenar** solo adaptadores LoRA pequeños.

La idea central es congelar el modelo cuantizado y aprender una corrección de bajo rango sobre cada matriz relevante.

Si una capa lineal original usa un peso $W \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$, LoRA introduce:

$$
W' = W + \Delta W, \qquad \Delta W = B A
$$

con

$$
A \in \mathbb{R}^{r \times d_{\text{in}}}, \qquad
B \in \mathbb{R}^{d_{\text{out}} \times r}, \qquad r \ll \min(d_{\text{in}}, d_{\text{out}})
$$

En lugar de entrenar todos los parámetros de $W$, solo se entrenan $A$ y $B$.

#### **Idea de memoria**

Si un modelo tiene $N$ parámetros y se almacena en precisión de 16 bits, el costo aproximado del peso base es:

$$
\text{Memoria base} \approx 2N \text{ bytes}
$$

Si el modelo se cuantiza a 4 bits, el costo ideal baja a:

$$
\text{Memoria cuantizada} \approx \frac{4}{8}N = 0.5N \text{ bytes}
$$

Luego, LoRA agrega un número pequeño de parámetros entrenables:

$$
\#\text{params LoRA} = r(d_{\text{in}} + d_{\text{out}})
$$

Por eso QLoRA permite ajustar modelos grandes con memoria muy inferior a full fine-tuning.

En la práctica hay costos extra por escalas, metadatos, estados del optimizador y activaciones, pero la intuición base es esa.

In [6]:
def memory_bytes(num_params, bits):
    return num_params * bits / 8

num_params = 7_000_000_000

for bits in [16, 8, 4]:
    gb = memory_bytes(num_params, bits) / (1024**3)
    print(f"{bits} bits -> {gb:.2f} GB solo para pesos")

16 bits -> 13.04 GB solo para pesos
8 bits -> 6.52 GB solo para pesos
4 bits -> 3.26 GB solo para pesos


#### **Cuantización de 4 bits y NF4**

QLoRA usa una cuantización de 4 bits diseñada para preservar mejor la información estadística de los pesos.

La idea general es aproximar un peso $w$ por un valor cuantizado $\hat{w}$ perteneciente a un conjunto discreto de niveles:

$$
\hat{w} = Q(w)
$$

Una forma simple de ver el error es:

$$
\varepsilon = w - \hat{w}
$$

y el objetivo implícito es que el error promedio sea pequeño bajo la distribución real de los pesos.

En QLoRA aparecen dos conceptos importantes:

1. **NF4**  
   Un formato de 4 bits pensado para pesos aproximadamente normales.

2. **Double Quantization**  
   También se cuantizan algunos parámetros auxiliares de la propia cuantización, lo que reduce aún más memoria.

En un laboratorio de clase no necesitas implementar NF4 desde cero para captar la idea, pero sí debes entender que el modelo base queda comprimido y congelado.

#### **Ejemplo de cuantización uniforme elemental**

Este ejemplo no reproduce NF4 exacto.  
Solo muestra cómo una cuantización de baja precisión aproxima una matriz real.

In [7]:
def uniform_quantize_4bit(x):
    x_min = x.min()
    x_max = x.max()
    levels = 16
    scale = (x_max - x_min) / (levels - 1 + 1e-12)
    q = torch.round((x - x_min) / (scale + 1e-12)).clamp(0, levels - 1)
    x_hat = q * scale + x_min
    return q, x_hat, scale, x_min

W = torch.randn(6, 6)
q, W_hat, scale, offset = uniform_quantize_4bit(W)

mse = F.mse_loss(W_hat, W).item()
print("MSE de reconstrucción:", round(mse, 6))
print("W[0]:", W[0])
print("W_hat[0]:", W_hat[0])

MSE de reconstrucción: 0.004312
W[0]: tensor([-0.8201,  0.3956,  0.8989, -1.3884, -0.1670,  0.2851])
W_hat[0]: tensor([-0.9243,  0.5010,  0.9761, -1.3995, -0.2117,  0.2635])


# Notas Devuelve:

q: los enteros de 4 bits (0–15).
x_hat: los valores reconstruidos.
scale: cuánto vale cada nivel.
x_min: el desplazamiento usado para reconstruir.

#### **LoRA mínima sobre una capa lineal**

La corrección LoRA suele escribirse como:

$$
h = xW^\top + x(\Delta W)^\top, \qquad \Delta W = \frac{\alpha}{r}BA
$$

donde $\alpha$ es un factor de escala.

En implementación, lo normal es:
- congelar $W$,
- inicializar $A$ y $B$,
- entrenar solo $A$ y $B$.

In [8]:
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=4, alpha=8):
        super().__init__()
        self.base = nn.Linear(in_features, out_features, bias=False)
        self.base.weight.requires_grad = False #congela pesos

        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.02) # matriz A
        self.B = nn.Parameter(torch.zeros(out_features, rank)) # matriz B

    def forward(self, x):
        base_out = self.base(x)
        lora_update = (x @ self.A.t()) @ self.B.t()
        return base_out + self.scaling * lora_update


layer = LoRALinear(16, 8, rank=2, alpha=4)
x = torch.randn(3, 16)
y = layer(x)

trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
total = sum(p.numel() for p in layer.parameters())

print("salida:", y.shape)
print("parámetros entrenables:", trainable)
print("parámetros totales:", total)
print("porcentaje entrenable:", round(100 * trainable / total, 2), "%")

salida: torch.Size([3, 8])
parámetros entrenables: 48
parámetros totales: 176
porcentaje entrenable: 27.27 %


#### **Qué se entrena en QLoRA**

En QLoRA, de forma conceptual:

- el modelo base cuantizado queda congelado,
- las capas LoRA sí reciben gradientes,
- el optimizador actualiza solo adaptadores,
- las activaciones siguen siendo un costo importante.

En otras palabras, si el modelo base tiene pesos $W_q$ cuantizados y la corrección entrenable es $\Delta W$, el entrenamiento actúa sobre:

$$
W_{\text{efectivo}} = W_q + \Delta W
$$

pero los gradientes solo actualizan $\Delta W$.

In [9]:
def count_parameters(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_parameters(layer)
print("total:", total)
print("trainable:", trainable)
print("frozen:", total - trainable)

total: 176
trainable: 48
frozen: 128


#### **Plantilla práctica con Transformers, bitsandbytes y PEFT**

Esta celda muestra la estructura típica de QLoRA.  
Está pensada como plantilla para adaptar luego a tu hardware y a tu dataset.

Puntos a notar:

- `load_in_4bit=True` carga el modelo base cuantizado.
- `bnb_4bit_quant_type="nf4"` usa el formato típico de QLoRA.
- `get_peft_model(...)` inserta adaptadores LoRA.
- el entrenamiento puede hacerse con `Trainer` o `SFTTrainer`.

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Cambia a True si quieres forzar CPU aunque haya GPU
USE_CPU = False

use_cuda = torch.cuda.is_available() and not USE_CPU

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

if use_cuda:
    print("Usando GPU con cuantización 4-bit")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    modelo = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )

    modelo = prepare_model_for_kbit_training(modelo)

else:
    print("Usando CPU sin cuantización 4-bit")

    modelo = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    )

    modelo.to("cpu")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

modelo = get_peft_model(modelo, lora_config)
modelo.print_trainable_parameters()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Usando CPU sin cuantización 4-bit


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


#### **Checklist conceptual de QLoRA**

Debes poder explicar estas preguntas:

1. ¿Por qué cuantizar a 4 bits reduce memoria?,
2. ¿Por qué LoRA evita entrenar todos los pesos?,
3. ¿Qué papel juegan $r$ y $\alpha$?,
4. ¿Qué trade-off aparece entre ahorro de memoria y calidad final?,
5. ¿Por qué QLoRA suele ser preferible a full fine-tuning en hardware limitado?.

### **Reward models y RLHF clásico**

Antes de DPO y ORPO conviene recordar el pipeline clásico de alineamiento con preferencias humanas.


#### **De preferencias humanas a reward model**

Supón que para un mismo `prompt` tenemos dos respuestas:

- una **chosen**,
- una **rejected**.

Un **reward model** intenta asignar un puntaje mayor a la respuesta preferida.  
Si llamamos $r_\phi(x, y)$ al puntaje del reward model, una pérdida de ranking sencilla puede escribirse como:

$$
\mathcal{L}_{\mathrm{RM}}(\phi)
=
-\log \sigma\left(r_\phi(x, y^{+}) - r_\phi(x, y^{-})\right)
$$

donde $y^{+}$ es la respuesta preferida y $y^{-}$ la rechazada.


#### **Ejemplo mínimo de preferencia para reward modeling**

Aquí no usamos texto real ni un LLM completo.  
Solo simulamos puntajes de un reward model para ver la lógica de la pérdida.


In [ ]:
def reward_ranking_loss(score_chosen, score_rejected):
    diff = score_chosen - score_rejected
    return -F.logsigmoid(diff).mean()

score_chosen = torch.tensor([2.1, 1.4, 0.8], dtype=torch.float32)
score_rejected = torch.tensor([0.7, 1.0, 0.6], dtype=torch.float32)

loss_rm = reward_ranking_loss(score_chosen, score_rejected)
print("Pérdida reward model:", float(loss_rm))
print("Diferencias:", (score_chosen - score_rejected).tolist())


#### **Pipeline clásico de RLHF**

Un pipeline clásico de **RLHF** suele verse así:

1. **SFT** sobre instrucciones o demostraciones.
2. **Reward model** entrenado con pares de preferencias.
3. **Optimización de la política** para maximizar recompensa, muchas veces con PPO.

Es decir:

$$
\text{Pretrained model}
\rightarrow
\text{SFT}
\rightarrow
\text{Reward Model}
\rightarrow
\text{PPO / RLHF}
$$

Este pipeline es potente, pero también introduce más piezas, más costo y más posibles fuentes de inestabilidad.


#### **Por qué DPO y ORPO resultan atractivos**

- **DPO** evita una etapa explícita de RL sobre la política y usa una comparación directa entre `chosen` y `rejected`.
- **ORPO** integra la preferencia junto con el objetivo supervisado en una sola función.

Por eso, en entornos docentes y proyectos medianos, DPO y ORPO suelen ser más sencillos de explicar, reproducir y defender que un pipeline RLHF completo.


### **DPO**

DPO usa pares de preferencias del tipo:

- `prompt`
- `chosen`
- `rejected`

La idea es empujar al modelo a dar mayor probabilidad a la respuesta preferida que a la rechazada, sin entrenar un reward model separado.

Si $\pi_\theta$ es la política actual y $\pi_{\text{ref}}$ una política de referencia, la pérdida de DPO puede escribirse como:

$$
\mathcal{L}_{\text{DPO}}(\theta) =
-\mathbb{E}\left[
\log \sigma \left(
\beta
\left(
\log \pi_\theta(y_w \mid x) - \log \pi_\theta(y_l \mid x)
-
\log \pi_{\text{ref}}(y_w \mid x) + \log \pi_{\text{ref}}(y_l \mid x)
\right)
\right)
\right]
$$

donde:
- $y_w$ es la respuesta preferida,
- $y_l$ es la respuesta rechazada,
- $\beta$ controla la intensidad de la preferencia.

#### **Lectura intuitiva**

DPO compara dos diferencias:

1. cuánto prefiere el modelo actual a la respuesta ganadora frente a la perdedora,
2. cuánto prefería ya el modelo de referencia esa misma pareja.

Luego aplica una sigmoide logística para premiar que la separación a favor de la respuesta correcta aumente.

Si definimos

$$
\Delta_\theta =
\log \pi_\theta(y_w \mid x) - \log \pi_\theta(y_l \mid x)
$$

y

$$
\Delta_{\text{ref}} =
\log \pi_{\text{ref}}(y_w \mid x) - \log \pi_{\text{ref}}(y_l \mid x)
$$

entonces DPO trabaja sobre:

$$
-\log \sigma\big(\beta(\Delta_\theta - \Delta_{\text{ref}})\big)
$$

Con esto, el modelo aprende preferencias relativas.

#### **Ejemplo para la pérdida DPO**

En lugar de usar un LLM completo, simularemos log-probabilidades ya calculadas.  
Así se ve con claridad qué hace la función objetivo.

In [ ]:
def dpo_loss(logp_chosen, logp_rejected, logp_ref_chosen, logp_ref_rejected, beta=0.1):
    pi_logratios = logp_chosen - logp_rejected
    ref_logratios = logp_ref_chosen - logp_ref_rejected
    logits = beta * (pi_logratios - ref_logratios)
    return -F.logsigmoid(logits).mean()

logp_chosen = torch.tensor([-2.0, -1.2, -0.9], requires_grad=True)
logp_rejected = torch.tensor([-2.5, -1.8, -1.1], requires_grad=True)

logp_ref_chosen = torch.tensor([-2.1, -1.3, -1.0])
logp_ref_rejected = torch.tensor([-2.3, -1.6, -1.05])

loss = dpo_loss(logp_chosen, logp_rejected, logp_ref_chosen, logp_ref_rejected, beta=0.5)
loss.backward()

print("Perdida DPO:", round(loss.item(), 6))
print("chosen gradiente :", logp_chosen.grad)
print("reject gradiente:", logp_rejected.grad)

#### **Qué aprende DPO**

Observa la dirección esperada:

- si la respuesta elegida ya es claramente mejor, la pérdida baja,
- si la respuesta rechazada tiene probabilidad alta, la pérdida sube,
- la política de referencia sirve como ancla para evitar desviaciones arbitrarias.

Por eso DPO suele ser más simple que RLHF con PPO:

- no necesitas reward model explícito,
- no necesitas una fase separada de PPO,
- el entrenamiento se parece más a una rutina supervisada con pares de preferencias.

#### **Formato típico del dataset de preferencias**

Cada muestra suele contener tres campos:

```python
{
    "prompt": "...",
    "chosen": "...",
    "rejected": "..."
}
```

Si tu dataset tiene otra estructura, debes transformarlo antes de usar `DPOTrainer`.

In [ ]:
# Ejemplo simple de dataset en memoria
preference_batch = [
    {
        "prompt": "Explica qué es LoRA.",
        "chosen": "LoRA aprende matrices de bajo rango y congela el peso base.",
        "rejected": "LoRA entrena todos los parámetros del modelo desde cero."
    },
    {
        "prompt": "¿Para qué sirve QLoRA?",
        "chosen": "Sirve para ajustar modelos grandes con muy poca memoria.",
        "rejected": "Sirve solo para acelerar inferencia y no para entrenamiento."
    }
]

for row in preference_batch:
    print(row["prompt"])
    print(" chosen  ->", row["chosen"])
    print(" rejected->", row["rejected"])
    print()

#### **Plantilla práctica con TRL para DPO**

La estructura base suele ser esta:

1. cargar modelo,
2. cargar modelo de referencia,
3. cargar tokenizer,
4. preparar dataset con `prompt`, `chosen`, `rejected`,
5. entrenar con `DPOTrainer`.

In [ ]:
# Plantilla orientativa para DPO.
# !pip install transformers datasets peft trl accelerate bitsandbytes

from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import DPOTrainer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

modelo = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
ref_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

training_args = TrainingArguments(
    output_dir="dpo_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=10,
    bf16=torch.cuda.is_available(),
)

# train_dataset debe tener columnas: prompt, chosen, rejected
# trainer = DPOTrainer(
#     model=modelo,
#     ref_model=ref_model,
#     args=training_args,
#     train_dataset=train_dataset,
#     processing_class=tokenizer,
#     beta=0.1,
# )
# trainer.train()

#### **Errores comunes en DPO**

1. **Pares mal construidos**  
   Si `chosen` y `rejected` no reflejan una preferencia real, el modelo aprende ruido.

2. **Referencia demasiado mala o demasiado parecida**  
   El término relativo pierde sentido si la política de referencia no está bien elegida.

3. **Prompt truncado o tokenización inconsistente**  
   Cambios sutiles en formato pueden alterar mucho las log-probabilidades.

4. **Beta mal calibrado**  
   Si $\beta$ es muy alto, el entrenamiento puede volverse agresivo.

### **ORPO**

ORPO significa **Odds Ratio Preference Optimization**.

La intuición es entrenar con una sola función objetivo que mezcla:

- aprendizaje supervisado sobre la respuesta deseada,
- presión relativa para separar la respuesta buena de la mala.

Una forma simplificada de escribir la idea es:

$$
\mathcal{L}_{\text{ORPO}} =
\mathcal{L}_{\text{SFT}} + \lambda \, \mathcal{L}_{\text{pref}}
$$

donde $\mathcal{L}_{\text{pref}}$ usa una comparación tipo razón de probabilidades entre respuesta elegida y rechazada.

#### **Intuición con odds ratio**

Si una respuesta tiene probabilidad $p$, sus odds son:

$$
\text{odds}(p) = \frac{p}{1-p}
$$

y su log-odds es:

$$
\log \frac{p}{1-p}
$$

ORPO aprovecha esta intuición para reforzar que la respuesta elegida tenga una ventaja clara sobre la rechazada, mientras también mantiene un componente tipo SFT.

#### **Versión simplificada de la pérdida ORPO**

Para fines pedagógicos, podemos pensar el bloque de preferencia como una penalización sobre la diferencia:

$$
\Delta_{\text{ORPO}} =
\log \frac{p_\theta(y_w \mid x)}{1 - p_\theta(y_w \mid x)}
-
\log \frac{p_\theta(y_l \mid x)}{1 - p_\theta(y_l \mid x)}
$$

y luego usar una forma logística:

$$
\mathcal{L}_{\text{pref}} = -\log \sigma(\Delta_{\text{ORPO}})
$$

La implementación real en librerías puede expresarse en términos de log-probabilidades secuenciales del modelo, pero esta vista ya captura la intuición.

In [ ]:
def safe_log_odds_from_logp(logp):
    p = torch.exp(logp).clamp(1e-6, 1 - 1e-6)
    return torch.log(p / (1 - p))

def orpo_toy_loss(logp_chosen, logp_rejected, lambda_pref=1.0):
    sft_term = -logp_chosen.mean()
    pref_margin = safe_log_odds_from_logp(logp_chosen) - safe_log_odds_from_logp(logp_rejected)
    pref_term = -F.logsigmoid(pref_margin).mean()
    return sft_term + lambda_pref * pref_term

logp_chosen_orpo = torch.log(torch.tensor([0.72, 0.81, 0.68], requires_grad=True))
logp_rejected_orpo = torch.log(torch.tensor([0.41, 0.36, 0.44], requires_grad=True))

loss_orpo = orpo_toy_loss(logp_chosen_orpo, logp_rejected_orpo, lambda_pref=0.5)
loss_orpo.backward()

print("Perdida ORPO:", round(loss_orpo.item(), 6))

#### **Cómo se diferencia ORPO de DPO**

DPO:
- usa una política de referencia,
- aprende con pares de preferencias,
- se centra en una comparación relativa respecto a esa referencia.

ORPO:
- puede evitar una referencia separada,
- mezcla supervisión y preferencia en un solo objetivo,
- puede simplificar el pipeline de entrenamiento.

En una discusión oral, una forma buena de resumirlo es:

> DPO alinea comparando contra una política de referencia.  
> ORPO alinea empujando la respuesta correcta y castigando la incorrecta dentro de una pérdida unificada.

#### **Plantilla práctica con TRL para ORPO**

La estructura típica es análoga a DPO, pero usando `ORPOTrainer`.

In [ ]:
# Plantilla orientativa para ORPO.
# !pip install transformers datasets trl peft accelerate bitsandbytes

from trl import ORPOTrainer, ORPOConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

orpo_args = ORPOConfig(
    output_dir="orpo_out",
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_length=512,
    max_prompt_length=256,
    logging_steps=10,
    beta=0.1,
)

# train_dataset debe tener columnas: prompt, chosen, rejected
# orpo_trainer = ORPOTrainer(
#     model=modelo,
#     args=orpo_args,
#     train_dataset=train_dataset,
#     processing_class=tokenizer,
# )
# orpo_trainer.train()

#### **Cuándo elegir QLoRA, DPO u ORPO**

Piensa en capas del pipeline:

1. **QLoRA**  
   Es una técnica de **adaptación eficiente**.  
   Responde a: "¿cómo ajusto un modelo grande con poca memoria?"

2. **DPO**  
   Es una técnica de **alineamiento por preferencias** con referencia.  
   Responde a: "¿cómo incorporo preferencias humanas de forma más simple que PPO?"

3. **ORPO**  
   Es una técnica de **alineamiento unificado**.  
   Responde a: "¿cómo junto señal supervisada y preferencia en una sola pérdida?"

### **Comparación rápida**

#### **Tabla conceptual**

| Método | Tipo de problema | Datos necesarios | Modelo de referencia | Costo relativo | Idea principal |
|---|---|---|---|---|---|
| QLoRA | Adaptación eficiente | prompts y targets | No | Bajo a medio | Cuantizar base y entrenar adaptadores |
| DPO | Alineamiento por preferencias | prompt, chosen, rejected | Sí | Medio | Aumentar preferencia relativa frente a referencia |
| ORPO | Alineamiento unificado | prompt, chosen, rejected | No obligatorio | Medio | Unir SFT y preferencia en una sola pérdida |

#### **Resumen operativo**

- Usa **QLoRA** cuando el cuello de botella es memoria.
- Usa **DPO** cuando ya tienes pares de preferencias y quieres un pipeline claro.
- Usa **ORPO** cuando te interesa una receta compacta de ajuste más preferencia.

#### **Riesgos y preguntas de laboratorio**

1. **QLoRA**
   - ¿qué módulos conviene adaptar?
   - ¿qué rango $r$ usar?
   - ¿cuánta memoria real ahorras?

2. **DPO**
   - ¿qué tan confiables son las preferencias?
   - ¿cómo eliges $\beta$?
   - ¿cómo evalúas que la política no se desvíe demasiado?

3. **ORPO**
   - ¿qué peso das al término de preferencia?
   - ¿cómo separas mejora real de sobreajuste a un dataset pequeño?


#### **Pipeline mínimo razonable**

Una secuencia práctica para un mini proyecto sería:

1. cargar un modelo pequeño,
2. aplicar **QLoRA** para SFT ligero,
3. preparar un dataset de preferencias,
4. correr **DPO** o **ORPO**,
5. evaluar con:
   - ejemplos manuales,
   - métricas automáticas si aplica,
   - análisis de errores y casos fallidos.


### **Sesgos y límites del alineamiento**

Alinear un modelo no significa volverlo perfecto.  
Muchas veces solo estamos optimizando una señal aproximada de preferencia humana.


#### **Fuentes comunes de sesgo**

Algunas fuentes típicas de sesgo en datasets de preferencias son:

- **annotator bias**: distintos evaluadores prefieren estilos distintos,
- **sesgo de longitud**: respuestas más largas pueden parecer mejores aunque no lo sean,
- **sesgo de tono**: respuestas más seguras o formales pueden ganar aunque contengan errores,
- **cobertura insuficiente**: pocos prompts o dominios limitan la generalización,
- **preferencias inconsistentes**: pares mal rotulados introducen ruido.

Esto afecta tanto a reward models como a DPO y ORPO.


#### **Límites prácticos**

Incluso cuando el entrenamiento funciona, pueden aparecer varios problemas:

1. **reward hacking** o sobreoptimización de la señal,
2. **colapso de diversidad**,
3. **sobre-refusal** en tareas donde el modelo debería responder,
4. **mejora aparente** en evaluación corta pero degradación fuera de distribución,
5. **alineamiento superficial** que mejora estilo, pero no necesariamente veracidad.

Por eso no basta con mirar una sola métrica.


### **Evaluación y análisis experimental**



#### **Qué medir antes y después**

Un bloque mínimo de evaluación puede revisar:

- **win rate** de la respuesta preferida,
- **longitud media** de la salida,
- **tasa de formato correcto**,
- **consistencia** de la respuesta,
- **costo de memoria**,
- **tiempo por paso**,
- **casos donde el modelo empeora**.

Si el objetivo es alignment, la pregunta clave es:

> ¿El modelo realmente mejora según la preferencia deseada, o solo cambia el estilo superficial?


#### **Ejemplo ligero de evaluación de preferencias**

La siguiente celda calcula una métrica muy simple: cuántas veces el modelo asigna mayor puntaje a `chosen` que a `rejected`.


In [ ]:
toy_eval = [
    {"chosen_score": 1.8, "rejected_score": 1.1},
    {"chosen_score": 0.9, "rejected_score": 1.2},
    {"chosen_score": 2.4, "rejected_score": 1.5},
    {"chosen_score": 1.7, "rejected_score": 1.6},
]

wins = sum(item["chosen_score"] > item["rejected_score"] for item in toy_eval)
total = len(toy_eval)
win_rate = wins / total

chosen_mean = sum(item["chosen_score"] for item in toy_eval) / total
rejected_mean = sum(item["rejected_score"] for item in toy_eval) / total

print(f"Win rate de chosen: {win_rate:.2%}")
print(f"Score medio chosen: {chosen_mean:.3f}")
print(f"Score medio rejected: {rejected_mean:.3f}")


#### **Análisis de errores que deberías poder defender**

Cuando presentes resultados, intenta separar al menos tres tipos de error:

- **error de datos**: pares mal etiquetados o prompts mal definidos,
- **error de optimización**: hiperparámetros, `beta`, `lambda`, `rank`, `alpha`,
- **error de evaluación**: métricas pobres o conjunto de prueba poco representativo.

Esta separación suele mejorar mucho la sustentación oral.


### **Ejercicios**

Los siguientes ejercicios buscan reforzar comprensión conceptual, cálculo rápido y lectura crítica de resultados.


#### **Ejercicios básicos**

1. Calcula la memoria aproximada de un modelo de **3B**, **7B** y **13B** parámetros en **16 bits**, **8 bits** y **4 bits**.

2. Para una capa lineal con $d_{\text{in}} = 1024$, $d_{\text{out}} = 4096$ y rango $r = 8$, calcula cuántos parámetros entrenables añade LoRA.

3. Explica en tus palabras por qué **QLoRA** puede ajustar un modelo grande sin actualizar todos sus pesos base.

4. Supón que en DPO tienes:
   - $\log p_\theta(y^+) = -1.2$
   - $\log p_\theta(y^-) = -2.1$
   - $\log p_{\mathrm{ref}}(y^+) = -1.5$
   - $\log p_{\mathrm{ref}}(y^-) = -1.8$

   Decide si la preferencia del modelo actual es más fuerte o más débil que la de referencia.

5. Explica la diferencia conceptual entre:
   - **SFT**
   - **DPO**
   - **ORPO**


#### **Ejercicios intermedios**

6. Modifica el `rank` en la clase `LoRALinear` con valores `2`, `4`, `8` y `16`.  
   Compara el porcentaje de parámetros entrenables.

7. Cambia `alpha` en LoRA y comenta cómo afecta la magnitud del update.

8. Cambia el tensor del ejemplo de cuantización y compara el error cuadrático medio para distintas distribuciones de valores.

9. En la pérdida DPO, prueba varios valores de `beta`: `0.05`, `0.1`, `0.5`, `1.0`.  
   Describe qué cambia numéricamente.

10. En la pérdida ORPO, modifica `lam` y explica cuándo el término de preferencia domina demasiado al objetivo supervisado.


#### **Ejercicios de análisis y sustentación**

11. Construye una tabla comparando **SFT**, **QLoRA**, **DPO**, **ORPO** y **RLHF** en estas columnas:
    - objetivo,
    - datos requeridos,
    - costo,
    - complejidad,
    - riesgo principal.

12. Diseña un mini dataset de **5 pares** `prompt / chosen / rejected` para un asistente académico.  
    Luego agrega **2 pares malos** y explica por qué introducirían ruido.

13. Elige un caso de uso:
    - tutor de programación,
    - chatbot universitario,
    - asistente de resumen de papers.

    Justifica si usarías:
    - **SFT + QLoRA**,
    - **QLoRA + DPO**,
    - **QLoRA + ORPO**,
    - o un pipeline **RLHF** más clásico.

14. Responde oralmente:
    - ¿por qué DPO necesita una referencia?
    - ¿por qué ORPO puede evitar una referencia explícita?
    - ¿qué trade-off introduce QLoRA?
    - ¿qué sesgos pueden aparecer en preferencias humanas?


#### **Ejercicio opcional de extensión**

Propón una mini práctica reproducible con estas piezas:

1. un modelo pequeño,
2. un dataset corto de instrucciones,
3. adaptación con **QLoRA**,
4. un conjunto simple de pares `chosen/rejected`,
5. ajuste con **DPO** o **ORPO**,
6. evaluación con `win rate` y análisis cualitativo.

La meta no es batir un benchmark.  
La meta es poder **explicar técnicamente** cada decisión tomada.
